In [1]:
import json
from IPython.display import HTML, display

# 1. Dados Iniciais Simulados (Backend Python)
dados_iniciais = [
    {"id": 1, "nome": "Carlos Eduardo", "idade": 45, "sintomas": "Dor intensa no peito irradiando para o braço esquerdo.", "classificacao": "VERMELHO"},
    {"id": 2, "nome": "Luiza de Deus", "idade": 22, "sintomas": "Febre alta constante (39°C) acompanhada de rigidez na nuca.", "classificacao": "LARANJA"},
    {"id": 3, "nome": "Warlley dos Santos", "idade": 29, "sintomas": "Crise asmática moderada com chiado no peito.", "classificacao": "AMARELO"},
    {"id": 4, "nome": "Matheus Oliveira", "idade": 19, "sintomas": "Sintomas gripais leves sem falta de ar.", "classificacao": "VERDE"}
]

dados_json = json.dumps(dados_iniciais, ensure_ascii=False)

# 2. Interface Integrada com Temporizador e Painel de Chamada (Tema Clínico Claro)
interface_colab = """
<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background-color: #f0f4f8; /* Fundo cinza/azul bem claro */
            color: #2d3748; /* Texto escuro e suave para leitura */
            margin: 0;
            padding: 15px;
        }
        .app-container { max-width: 1000px; margin: 0 auto; }
        .app-header {
            text-align: center; padding-bottom: 15px;
            border-bottom: 2px solid #e2e8f0; margin-bottom: 20px;
        }
        .app-header h1 { color: #2b6cb0; margin: 0; font-size: 1.8rem; }
        .app-header p { color: #718096; }

        /* Painel de Chamada em Destaque */
        .call-panel {
            background-color: #ffffff;
            border: 2px solid #4299e1;
            border-radius: 8px;
            padding: 20px;
            text-align: center;
            margin-bottom: 20px;
            box-shadow: 0 4px 15px rgba(66, 153, 225, 0.15);
        }
        .call-panel h3 { margin: 0 0 10px 0; color: #718096; font-size: 1rem; text-transform: uppercase; }
        .current-patient { font-size: 2.2rem; font-weight: bold; color: #2b6cb0; margin-bottom: 5px; }
        .current-room { font-size: 1.2rem; color: #38a169; }
        .timer-badge {
            display: inline-block;
            background-color: #ebf8ff;
            border: 1px solid #bee3f8;
            padding: 5px 12px;
            border-radius: 15px;
            font-size: 0.9rem;
            color: #2b6cb0;
            margin-top: 15px;
            font-weight: bold;
        }

        .workspace { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }
        .card-panel { background-color: #ffffff; padding: 22px; border-radius: 8px; border: 1px solid #e2e8f0; box-shadow: 0 2px 4px rgba(0,0,0,0.05); }
        .card-panel h2 { margin-top: 0; font-size: 1.2rem; border-left: 4px solid #4299e1; padding-left: 10px; color: #2d3748; }

        .form-group { display: flex; flex-direction: column; margin-bottom: 12px; }
        label { margin-bottom: 5px; color: #4a5568; font-size: 0.85rem; font-weight: 600; }
        input, textarea { padding: 10px; background-color: #ffffff; border: 1px solid #cbd5e0; border-radius: 4px; color: #2d3748; }
        input:focus, textarea:focus { border-color: #4299e1; outline: none; box-shadow: 0 0 0 2px rgba(66, 153, 225, 0.2); }

        /* Árvore de Decisão UI */
        .decision-tree-panel { background-color: #f7fafc; border: 1px dashed #4299e1; padding: 15px; border-radius: 6px; margin-bottom: 15px; }
        .question-text { font-size: 1rem; margin-bottom: 12px; font-weight: bold; color: #2d3748; }
        .tree-buttons { display: flex; gap: 10px; }
        .btn-tree { flex: 1; padding: 8px; border: none; border-radius: 4px; cursor: pointer; font-weight: bold; }
        .btn-sim { background-color: #38a169; color: white; }
        .btn-nao { background-color: #e53e3e; color: white; }

        .actions-bar { display: flex; gap: 10px; margin-top: 15px; }
        button { padding: 10px; border: none; border-radius: 4px; cursor: pointer; font-weight: bold; width: 100%; transition: background-color 0.2s; }
        .btn-save { background-color: #3182ce; color: #fff; }
        .btn-save:hover:not(:disabled) { background-color: #2b6cb0; }
        .btn-save:disabled { background-color: #e2e8f0; cursor: not-allowed; color: #a0aec0; }

        /* Cores Manchester - Ajustadas para contraste no tema claro */
        .VERMELHO { background-color: #e53e3e; color: #ffffff; }
        .LARANJA { background-color: #ed8936; color: #ffffff; }
        .AMARELO { background-color: #ecc94b; color: #2d3748; } /* Letra escura no amarelo para leitura */
        .VERDE { background-color: #38a169; color: #ffffff; }
        .AZUL { background-color: #3182ce; color: #ffffff; }

        .border-VERMELHO { border-left: 6px solid #e53e3e !important; }
        .border-LARANJA { border-left: 6px solid #ed8936 !important; }
        .border-AMARELO { border-left: 6px solid #ecc94b !important; }
        .border-VERDE { border-left: 6px solid #38a169 !important; }
        .border-AZUL { border-left: 6px solid #3182ce !important; }

        .patient-item {
            background-color: #ffffff; padding: 12px; border-radius: 6px; margin-bottom: 10px;
            border: 1px solid #e2e8f0; transition: transform 0.2s; display: flex; justify-content: space-between; align-items: center;
        }
        .empty-list { text-align: center; color: #a0aec0; padding: 20px; font-style: italic; }
    </style>
</head>
<body>

<div class="app-container">
    <div class="app-header">
        <h1>Sistema Central de Triagem</h1>
        <p>Atendimento Automático Inteligente com Árvore de Decisão</p>
    </div>

    <div class="call-panel" id="call-panel">
        <h3>Sendo atendido agora</h3>
        <div class="current-patient" id="display-nome">Aguardando...</div>
        <div class="current-room" id="display-classificacao">Nenhum paciente chamado</div>
        <div class="timer-badge" id="timer-display">⏱️ Próxima chamada em: 10s</div>
    </div>

    <div class="workspace">
        <div class="card-panel">
            <h2>Cadastrar Novo Paciente</h2>
            <form id="crud-form" onsubmit="processarFormulario(event)">
                <input type="hidden" id="final-classification" required>

                <div style="display: flex; gap: 10px;">
                    <div class="form-group" style="flex: 2;">
                        <label>Nome Completo</label>
                        <input type="text" id="field-nome" required>
                    </div>
                    <div class="form-group" style="flex: 1;">
                        <label>Idade</label>
                        <input type="number" id="field-idade" required>
                    </div>
                </div>

                <div class="form-group">
                    <label>Sintomas</label>
                    <textarea id="field-sintomas" rows="2" required></textarea>
                </div>

                <div class="decision-tree-panel">
                    <label style="color: #3182ce;">Motor de Inferência</label>
                    <div class="question-text" id="tree-question">Iniciando avaliação...</div>
                    <div class="tree-buttons" id="tree-controls">
                        <button type="button" class="btn-tree btn-sim" onclick="responderArvore('sim')">Sim</button>
                        <button type="button" class="btn-tree btn-nao" onclick="responderArvore('nao')">Não</button>
                    </div>
                </div>

                <button type="submit" class="btn-save" id="btn-submit" disabled>Concluir Triagem para Inserir</button>
            </form>
        </div>

        <div class="card-panel">
            <h2>Fila de Espera Dinâmica</h2>
            <div id="lista-atendimento"></div>
        </div>
    </div>
</div>

<script>
    let basePacientes = __DADOS__;
    const hierarquia = { "VERMELHO": 5, "LARANJA": 4, "AMARELO": 3, "VERDE": 2, "AZUL": 1 };

    // --- LÓGICA DO TEMPORIZADOR E CHAMADA AUTOMÁTICA ---
    let tempoRestante = 10;

    setInterval(() => {
        const displayTimer = document.getElementById("timer-display");

        if (basePacientes.length > 0) {
            tempoRestante--;
            displayTimer.innerText = `⏱️ Próxima chamada em: ${tempoRestante}s`;

            // Alterado para manter boa visualização no tema claro
            if (tempoRestante <= 3) {
                displayTimer.style.color = "#e53e3e";
                displayTimer.style.backgroundColor = "#fed7d7";
                displayTimer.style.borderColor = "#feb2b2";
            } else {
                displayTimer.style.color = "#2b6cb0";
                displayTimer.style.backgroundColor = "#ebf8ff";
                displayTimer.style.borderColor = "#bee3f8";
            }

            if (tempoRestante <= 0) {
                chamarProximoPaciente();
                tempoRestante = 10; // Reinicia o timer
            }
        } else {
            displayTimer.innerText = "⏱️ Fila vazia. Aguardando registros...";
            displayTimer.style.color = "#718096";
            displayTimer.style.backgroundColor = "#f7fafc";
            displayTimer.style.borderColor = "#e2e8f0";
            tempoRestante = 10;
        }
    }, 1000);

    function chamarProximoPaciente() {
        if (basePacientes.length === 0) return;

        // Ordena: Primeiro por Gravidade (decrescente), depois por Ordem de Chegada (ID crescente)
        basePacientes.sort((a, b) => {
            if (hierarquia[b.classificacao] !== hierarquia[a.classificacao]) {
                return hierarquia[b.classificacao] - hierarquia[a.classificacao];
            }
            return a.id - b.id;
        });

        const pacienteChamado = basePacientes.shift();

        // Atualiza o painel principal
        const panel = document.getElementById("call-panel");
        panel.style.borderColor = getCorHex(pacienteChamado.classificacao);

        document.getElementById("display-nome").innerText = pacienteChamado.nome;
        document.getElementById("display-nome").style.color = getCorHex(pacienteChamado.classificacao);

        document.getElementById("display-classificacao").innerText = `Prioridade: ${pacienteChamado.classificacao} - Dirija-se ao consultório.`;

        renderizarFila();
    }

    function getCorHex(cor) {
        const cores = {
            "VERMELHO": "#e53e3e", "LARANJA": "#ed8936",
            "AMARELO": "#ecc94b", "VERDE": "#38a169", "AZUL": "#3182ce"
        };
        return cores[cor];
    }
    // ---------------------------------------------------

    // --- ÁRVORE DE DECISÃO ---
    const arvoreManchester = {
        pergunta: "1. Comprometimento de vias aéreas, parada cardiorrespiratória ou choque?",
        sim: "VERMELHO",
        nao: {
            pergunta: "2. Dor severa, hemorragia incontrolável ou alteração de consciência?",
            sim: "LARANJA",
            nao: {
                pergunta: "3. Dor moderada, febre alta persistente ou dificuldade respiratória?",
                sim: "AMARELO",
                nao: {
                    pergunta: "4. Dor leve ou sintomas recentes causando desconforto?",
                    sim: "VERDE",
                    nao: "AZUL"
                }
            }
        }
    };

    let noAtual = null;

    function iniciarArvore() {
        noAtual = arvoreManchester;
        document.getElementById("tree-question").innerText = noAtual.pergunta;
        document.getElementById("tree-controls").style.display = "flex";
        document.getElementById("final-classification").value = "";
        document.getElementById("btn-submit").disabled = true;
        document.getElementById("btn-submit").innerText = "Responda à triagem para liberar";
    }

    function responderArvore(resposta) {
        const proximoPasso = noAtual[resposta];
        if (typeof proximoPasso === "string") {
            finalizarTriagem(proximoPasso);
        } else {
            noAtual = proximoPasso;
            document.getElementById("tree-question").innerText = noAtual.pergunta;
        }
    }

    function finalizarTriagem(cor) {
        document.getElementById("tree-controls").style.display = "none";
        document.getElementById("tree-question").innerHTML = `Classificação Calculada: <span class="${cor}" style="padding: 2px 8px; border-radius: 4px;">${cor}</span>`;
        document.getElementById("final-classification").value = cor;
        document.getElementById("btn-submit").disabled = false;
        document.getElementById("btn-submit").innerText = "Inserir na Fila";
    }

    // --- RENDERIZAÇÃO E FORMULÁRIO ---
    function renderizarFila() {
        const container = document.getElementById("lista-atendimento");
        container.innerHTML = "";

        if (basePacientes.length === 0) {
            container.innerHTML = '<div class="empty-list">Nenhum paciente aguardando atendimento.</div>';
            return;
        }

        const filaOrdenada = [...basePacientes].sort((a, b) => {
            if (hierarquia[b.classificacao] !== hierarquia[a.classificacao]) {
                return hierarquia[b.classificacao] - hierarquia[a.classificacao];
            }
            return a.id - b.id;
        });

        filaOrdenada.forEach(paciente => {
            const item = document.createElement("div");
            item.className = `patient-item border-${paciente.classificacao}`;
            item.innerHTML = `
                <div>
                    <div style="font-weight: bold; color: #2d3748;">${paciente.nome}</div>
                    <div style="font-size: 0.8rem; color: #718096;">${paciente.sintomas}</div>
                </div>
                <span class="${paciente.classificacao}" style="font-size: 0.75rem; padding: 4px 8px; border-radius: 4px; font-weight: bold;">${paciente.classificacao}</span>
            `;
            container.appendChild(item);
        });
    }

    function processarFormulario(event) {
        event.preventDefault();
        const nome = document.getElementById("field-nome").value;
        const idade = parseInt(document.getElementById("field-idade").value);
        const sintomas = document.getElementById("field-sintomas").value;
        const classificacao = document.getElementById("final-classification").value;

        const proximoId = basePacientes.length > 0 ? Math.max(...basePacientes.map(p => p.id)) + 1 : 1;
        basePacientes.push({ id: proximoId, nome, idade, sintomas, classificacao });

        document.getElementById("crud-form").reset();
        iniciarArvore();
        renderizarFila();
    }

    iniciarArvore();
    renderizarFila();
</script>
</body>
</html>
""".replace("__DADOS__", dados_json)

display(HTML(interface_colab))